In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import time

In [19]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 20000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
eval_iters = 200
n_embd = 384
n_head = 24
n_layer = 60
dropout = 0.2
# ------------

cpu


In [20]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#with open('code.txt', 'r', encoding='utf-8') as f:
#    text = f.read()
    
with open('rich_dad_poor_dad.txt', 'r', encoding='utf-8') as f:
    text = f.read()


# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest va
train_data = data[:n]
val_data = data[n:]

In [21]:

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

head_size = n_embd // n_head

global_query = nn.Linear(n_embd, head_size, bias=False)
global_value = nn.Linear(n_embd, head_size, bias=False)

class DeepSeekMLA(nn.Module):
    """ 
    DeepSeek Multi-Head Latent Attention (MLA) 
    Replaces MultiHeadAttention + Head classes
    """
    def __init__(self, n_embd, n_head, d_c=32, d_cq=32, d_rope=16):
        super().__init__()
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.d_c = d_c       # KV Compression dimension
        self.d_cq = d_cq     # Query Compression dimension
        self.d_rope = d_rope # RoPE dimension



        # KV Path: Down-projection to latent space (this is what you cache)
        self.kv_down_proj = nn.Linear(n_embd, d_c, bias=False)
        self.kv_up_proj = nn.Linear(d_c, n_head * (self.head_size + d_rope), bias=False)
        
        # Query Path: Low-rank compression for training efficiency
        self.q_down_proj = nn.Linear(n_embd, d_cq, bias=False)
        self.q_up_proj = nn.Linear(d_cq, n_head * (self.head_size + d_rope), bias=False)

        self.out_proj = nn.Linear(n_head * self.head_size, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        # Causal mask (tril)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        # 1. KV Latent Projection
        c_kv = self.kv_down_proj(x) # (B, T, d_c)
        kv_up = self.kv_up_proj(c_kv).view(B, T, self.n_head, self.head_size + self.d_rope)
        # Split into content Key and RoPE Key
        k_content, k_rope = kv_up.split([self.head_size, self.d_rope], dim=-1)
        v = k_content # In MLA, V is derived from the same latent vector

        # 2. Query Latent Projection
        c_q = self.q_down_proj(x) # (B, T, d_cq)
        q_up = self.q_up_proj(c_q).view(B, T, self.n_head, self.head_size + self.d_rope)
        q_content, q_rope = q_up.split([self.head_size, self.d_rope], dim=-1)

        # 3. Attention Calculation
        # Transpose for batch-head format: (B, nh, T, d)
        q_content = q_content.transpose(1, 2)
        k_content = k_content.transpose(1, 2)
        q_rope = q_rope.transpose(1, 2)
        k_rope = k_rope.transpose(1, 2)
        v = v.transpose(1, 2)

        # Compute content and RoPE attention scores separately then add
        # This is a simplified version of the decoupled RoPE
        wei = (q_content @ k_content.transpose(-2, -1) + 
               q_rope @ k_rope.transpose(-2, -1)) * (self.head_size + self.d_rope)**-0.5
        
        # Masking and Softmax
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # 4. Final Aggregation
        out = (wei @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.out_proj(out)

        
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Update the Block class to use the new MLA module
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # Replace sa with MLA
        self.sa = DeepSeekMLA(n_embd, n_head) 
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [22]:
model = GPTLanguageModel()
#model.load_state_dict(torch.load("./model_code"))
#model.load_state_dict(torch.load("./model_rich"))

m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# 1, 35.664477 M parameters
# 2, 42.730077 M parameters

84.427869 M parameters


In [23]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:

start_time = time.time()  # Record the start time
for iter in range(1000):
    
    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        end_time = time.time()  # Record the end time
        time_taken = end_time - start_time  # Calculate time taken

        print(f"Iteration {iter} took {time_taken:.4f} seconds")  # Print the time taken

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()